# Networking + Containerization: Overlay Networks and Service Mesh Exploration

This notebook combines Networking Fundamentals with Containerization Concepts to explore how overlay networks enable container communication across hosts and how service meshes add observability and traffic management on top of that network fabric.

## Why combine Networking with Containerization

Containerized applications rarely run on a single host. Once services are spread across multiple machines, the network becomes the invisible glue holding everything together. Docker's default bridge network isolates containers on one host, but overlay networks extend that connectivity across the cluster. On top of that, a service mesh (like Istio or Linkerd) layers traffic management, mutual TLS, and observability without changing application code.

Understanding this stack — from physical NICs to overlay encapsulation to mesh proxies — explains why a service in Container A can reach Container B on a different host as if they were on the same LAN, and how you get metrics, retries, and canary deployments for free.

## Key terminology

- **Overlay network** — a virtual network built on top of an existing network (underlay) that allows containers on different hosts to communicate as if they were on the same local network. Example: Docker Swarm creates overlay networks automatically.
- **Underlay network** — the physical or virtual network that hosts connect to. The overlay tunnels traffic through the underlay. Example: the host's eth0 network.
- **VXLAN** — Virtual Extensible LAN, the encapsulation protocol most overlay networks use. It wraps layer-2 Ethernet frames inside UDP packets to tunnel across the underlay. Example: Docker overlay networks use VXLAN by default.
- **Service mesh** — an infrastructure layer that handles service-to-service communication (load balancing, retries, mTLS, observability) via sidecar proxies injected alongside each service instance. Example: Istio, Linkerd.
- **Sidecar proxy** — a proxy process (like Envoy) deployed alongside each service instance that intercepts all inbound and outbound traffic. Example: Istio's Envoy sidecar.
- **mTLS (mutual TLS)** — both client and server authenticate each other via certificates. Service meshes use mTLS to encrypt all inter-service traffic automatically.
- **Traffic splitting** — routing a percentage of requests to a different service version. Example: sending 10% of traffic to a canary release.
- **Service discovery** — the mechanism by which services find each other's network locations. Example: Kubernetes DNS gives each service a predictable hostname.

## Overlay network fundamentals

An overlay network solves a simple problem: containers on different hosts need to communicate, but the underlay network only knows about host IPs. The overlay encapsulates container-to-container traffic inside host-to-host tunnels.

The process works like this:
1. Each host runs an overlay driver that maintains a VTEP (VXLAN Tunnel Endpoint)
2. When Container A on Host 1 sends a packet to Container B on Host 2, the VTEP on Host 1 wraps the packet in a VXLAN header with Host 2's IP as the outer destination
3. The underlay network routes the encapsulated packet to Host 2
4. Host 2's VTEP strips the VXLAN header and delivers the original packet to Container B

From the containers' perspective, they're on the same flat network — the encapsulation is invisible to them.

In [ ]:
"""
Overlay network concepts — exploring VXLAN encapsulation math.
This cell calculates the overhead overlay networks add to packets.
"""

# VXLAN adds 50 bytes of overhead per packet
# Original Ethernet frame: up to 1500 bytes (MTU)
# With VXLAN: the underlay needs MTU of at least 1550

ORIGINAL_MTU = 1500
VXLAN_OVERHEAD = 50  # bytes: outer Ethernet(14) + outer IP(20) + outer UDP(8) + VXLAN(8) + inner Ethernet(14) - FCS(-4)
UNDERLAY_MTU_NEEDED = ORIGINAL_MTU + VXLAN_OVERHEAD

print("=== VXLAN Overlay Overhead ===")
print(f"Original container MTU: {ORIGINAL_MTU} bytes")
print(f"VXLAN encapsulation overhead: {VXLAN_OVERHEAD} bytes")
print(f"Minimum underlay MTU needed: {UNDERLAY_MTU_NEEDED} bytes")
print()

# Typical MTU values for different environments
mtu_scenarios = {
    "Standard Ethernet": 1500,
    "Jumbo frame (data center)": 9000,
    "AWS VPC (typical)": 9001,
    "Cloud overlay-safe": 1450,  # accounts for additional cloud encapsulation
}

print("Can each MTU support VXLAN overlay?")
for name, mtu in mtu_scenarios.items():
    safe = mtu >= UNDERLAY_MTU_NEEDED
    status = "OK" if safe else "FRAGMENTATION RISK"
    print(f"  {name} (MTU {mtu}): {status}")

## Docker overlay network example

Docker makes overlay networks straightforward to create and use. Once a Swarm is initialized (or an external key-value store like Consul is configured), a single command creates a network that spans all nodes.

In [ ]:
"""
Docker overlay network — inspecting network properties via CLI.
These commands show how to create and inspect overlay networks.
"""

import subprocess
import json


def run_cmd(cmd):
    """Run a shell command and return stdout."""
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=10)
        return result.stdout.strip()
    except Exception as e:
        return f"Error: {e}"


def inspect_network(name):
    """Inspect a Docker network and return key properties."""
    raw = run_cmd(f"docker network inspect {name} 2>/dev/null")
    if raw.startswith("Error") or not raw:
        return None
    try:
        data = json.loads(raw)
        if data:
            net = data[0]
            return {
                "name": net.get("Name"),
                "driver": net.get("Driver"),
                "scope": net.get("Scope"),
                "subnet": net.get("IPAM", {}).get("Config", [{}])[0].get("Subnet"),
                "gateway": net.get("IPAM", {}).get("Config", [{}])[0].get("Gateway"),
            }
    except json.JSONDecodeError:
        pass
    return None


# List available networks
print("=== Docker Networks ===")
output = run_cmd("docker network ls --format '{{.Name}}\t{{.Driver}}\t{{.Scope}}' 2>/dev/null")
if output:
    print(output)
else:
    print("Docker not available or no networks found.")

print()

# Try to inspect the default bridge network
for net_name in ["bridge", "host", "overlay_test"]:
    info = inspect_network(net_name)
    if info:
        print(f"Network: {info['name']}")
        print(f"  Driver: {info['driver']}")
        print(f"  Scope: {info['scope']}")
        if info['subnet']:
            print(f"  Subnet: {info['subnet']}")
        print()

## Service mesh architecture

A service mesh adds a networking layer between services without modifying the services themselves. The key architectural pattern is the sidecar proxy: every service instance gets a lightweight proxy (usually Envoy) deployed alongside it.

The mesh handles:
- **Traffic management**: load balancing, retries, timeouts, circuit breaking
- **Security**: automatic mTLS between all services
- **Observability**: request-level metrics, distributed tracing, access logging
- **Policy**: rate limiting, authorization rules

This separation of concerns means application developers focus on business logic while platform engineers control the network behavior through mesh configuration.

In [ ]:
"""
Service mesh concepts — simulating traffic splitting between service versions.
This demonstrates the core idea behind canary deployments in a mesh.
"""

import random


def simulate_traffic_split(versions, weights, num_requests=100):
    """
    Simulate routing requests across service versions based on weights.
    This is how a service mesh implements traffic splitting.
    """
    results = {v: 0 for v in versions}
    
    for _ in range(num_requests):
        roll = random.random() * 100
        cumulative = 0
        for version, weight in zip(versions, weights):
            cumulative += weight
            if roll <= cumulative:
                results[version] += 1
                break
    
    return results


# Scenario: deploying v2 as a canary release
print("=== Service Mesh Traffic Splitting ===")
print("Scenario: v1 is stable, v2 is canary (5% traffic)")
print()

versions = ["v1 (stable)", "v2 (canary)"]
weights = [95, 5]  # percentage

results = simulate_traffic_split(versions, weights, num_requests=1000)
for version, count in results.items():
    pct = count / 10
    print(f"  {version}: {count} requests ({pct:.1f}%)")

print()
print("Scenario: gradual rollout — v2 gets 50%")
weights = [50, 50]
results = simulate_traffic_split(versions, weights, num_requests=1000)
for version, count in results.items():
    pct = count / 10
    print(f"  {version}: {count} requests ({pct:.1f}%)")

print()
print("In a real mesh (Istio/Linkerd), this is configured with a VirtualService")
print("or TrafficSplit resource — no application code changes needed.")

## Mesh observability: what the proxy sees

One of the biggest wins from a service mesh is observability. The sidecar proxy sees every request, which means you get metrics without instrumenting application code. The standard metrics the mesh exposes are the "golden signals":

- **Latency**: how long requests take (p50, p95, p99)
- **Traffic**: requests per second
- **Errors**: error rate (4xx, 5xx responses)
- **Saturation**: how full the service is (connection count, queue depth)

These map directly to Prometheus metrics that the mesh scrapes and makes queryable.

In [ ]:
"""
Mesh observability — computing golden signals from simulated request data.
"""

import random
import statistics


def generate_request_latencies(num_requests=500, base_ms=10, jitter_ms=40):
    """Simulate request latencies as a mesh proxy would observe them."""
    return [base_ms + random.expovariate(1.0 / jitter_ms) for _ in range(num_requests)]


def compute_golden_signals(latencies, error_rate_pct=2.0):
    """Compute the four golden signals from request data."""
    total_requests = len(latencies)
    errors = int(total_requests * error_rate_pct / 100)
    
    return {
        "total_requests": total_requests,
        "error_rate_pct": error_rate_pct,
        "errors": errors,
        "successful": total_requests - errors,
        "latency_p50_ms": statistics.median(latencies),
        "latency_p95_ms": sorted(latencies)[int(total_requests * 0.95)],
        "latency_p99_ms": sorted(latencies)[int(total_requests * 0.99)],
        "latency_mean_ms": statistics.mean(latencies),
    }


print("=== Mesh Golden Signals (simulated) ===")
latencies = generate_request_latencies()
signals = compute_golden_signals(latencies)

print(f"Total requests: {signals['total_requests']}")
print(f"Error rate: {signals['error_rate_pct']}% ({signals['errors']} errors)")
print(f"Latency p50: {signals['latency_p50_ms']:.1f}ms")
print(f"Latency p95: {signals['latency_p95_ms']:.1f}ms")
print(f"Latency p99: {signals['latency_p99_ms']:.1f}ms")
print(f"Latency mean: {signals['latency_mean_ms']:.1f}ms")

print()
print("These are the metrics Istio/Linkerd expose to Prometheus.")
print("A PromQL query for p99 latency in Istio would be:")
print('  histogram_quantile(0.99, sum(rate(istio_request_duration_milliseconds_bucket[5m])) by (le, destination_service))')

## Connecting overlay networks to the mesh

The overlay network and the service mesh are complementary layers:

1. **Overlay network** provides the connectivity — containers on different hosts can reach each other
2. **Service mesh** provides the intelligence — traffic management, security, and observability on top of that connectivity

In Kubernetes, the CNI plugin (Calico, Cilium, Flannel) creates the overlay or underlay network. Then the service mesh (Istio, Linkerd) injects sidecar proxies that intercept traffic flowing through that network.

The key insight: the mesh doesn't replace the network — it layers on top of it. Without a working overlay network, the mesh has nothing to manage.

In [ ]:
"""
Mapping the full network stack — from physical NIC to mesh proxy.
This shows the layers a request traverses in a containerized environment.
"""

layers = [
    ("Application", "Your service code", "HTTP/gRPC request"),
    ("Sidecar Proxy", "Envoy (mesh data plane)", "Intercepts all traffic, applies policies"),
    ("Container Network", "eth0 inside container", "Virtual NIC (veth pair)"),
    ("Overlay Network", "VXLAN tunnel", "Encapsulates cross-host traffic"),
    ("Underlay Network", "Host NIC (eth0, ens5)", "Physical/virtual network"),
    ("Physical Layer", "Switches, routers, cables", "Actual packets on the wire"),
]

print("=== Request Path: Container to Container (cross-host) ===")
print()
for i, (name, component, role) in enumerate(layers, 1):
    arrow = "  ↓" if i < len(layers) else "  ✓"
    print(f"{i}. {name:20s} │ {component:30s} │ {role}")
    if i < len(layers):
        print(f"   {'':20s} │ {'':30s} │")

print()
print("Each layer adds its own headers/encapsulation but is transparent to the layers above.")
print("The overlay hides cross-host complexity; the mesh hides traffic-management complexity.")

## How this connects to what's next

This exploration covers the conceptual foundation of overlay networks and service meshes. The next step would be hands-on experimentation: spinning up a Docker Swarm or kind cluster, creating overlay networks, and deploying a mesh like Linkerd to observe real traffic patterns. The observability metrics computed here map directly to Prometheus queries you will write when working with Grafana dashboards for service mesh monitoring.